# 00c · Data download (Milestone 2, before checkpoint CP1)

Fetches and verifies the CP1 datasets with `eviground.data.download.ensure_data()`, following the
storage rules of spec section 13 (revision 45). It replaces the retired downloader notebook.

| Key | What happens | Drive | Local disk |
|---|---|---|---|
| `genimage_val` | GenImage Validation: 5 generators already on Drive (loose). The missing GLIDE, SD v1.4 and VQDM are taken from the Kaggle Validation archive (≈ 25 GB, to local disk, then deleted) and stored as one tar file each; then all 8 generators and 100,000 images are verified | ≈ 10 GB | ≈ 25 GB |
| `diffseg30k` | `train.zip` + `validation.zip` copied to Drive as published, SHA-256 checked | ≈ 26.5 GB | Drive write cache |
| `sidset` | Already on Drive (10,000 per class train, 2,000 per class validation, from the retired downloader): verified in place, nothing downloaded | 0 | 0 |
| `genimage_sd14` | 89.8 GB archive to local disk (not extracted) → seeded 40,000 real + 40,000 generated → a few tar files on Drive → local archive deleted | ≈ 21–25 GB | ≈ 90 GB |

**Sessions** (fresh CPU runtime each):
1. Done on 12 Nov 2026: `diffseg30k` and `sidset` verified.
2. `DATASETS = ['genimage_val', 'genimage_sd14']`, in this order. Each archive is deleted from the local disk before the next download starts; the SD v1.4 archive needs about 90 GB of it, so nothing else should run in that session.

Every step checks free space on Drive and on the local disk first and stops with a message
if something will not fit. Every step resumes after a disconnect: re-run the same cell. `_DONE` is
written only after file counts, sizes and checksums are verified. A dataset with `_DONE` is
skipped with no download and no secret read.

**Secrets** (Colab Secrets panel, notebook access on): `GITHUB_TOKEN` (clone while private);
`HF_TOKEN` (needed if a dataset is gated); for `genimage_val` and `genimage_sd14`,
`KAGGLE_API_TOKEN`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`. They are read only inside the download calls, never printed or saved.

Result: `Paper 1/runs/checkpoints/cp1_download_<datasets>.json`. Afterwards run
`00b_data_probe.ipynb` again so the probe records what is now on Drive.

In [5]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)

sys.path.insert(0, os.path.join(REPO, 'src'))
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/Paper1 | branch: main | dry run: False


In [6]:
import sys; [sys.modules.pop(m) for m in list(sys.modules) if m.startswith('eviground')]

[<module 'eviground' from '/content/Paper1/src/eviground/__init__.py'>,
 <module 'eviground.data' from '/content/Paper1/src/eviground/data/__init__.py'>,
 <module 'eviground.data.registry' from '/content/Paper1/src/eviground/data/registry.py'>,
 <module 'eviground.data.storage' from '/content/Paper1/src/eviground/data/storage.py'>,
 <module 'eviground.env' from '/content/Paper1/src/eviground/env.py'>,
 <module 'eviground.data.imageinfo' from '/content/Paper1/src/eviground/data/imageinfo.py'>,
 <module 'eviground.data.sources' from '/content/Paper1/src/eviground/data/sources.py'>,
 <module 'eviground.data.download' from '/content/Paper1/src/eviground/data/download.py'>,
 <module 'eviground.data.probe' from '/content/Paper1/src/eviground/data/probe.py'>]

In [7]:
# Cell 2: choose the datasets for this session; show free space before anything starts
import json, logging, time
logging.basicConfig(level=logging.INFO, force=True)
from eviground.data.registry import REGISTRY
from eviground.data.storage import free_bytes
from eviground.env import get_paths, session_info

DATASETS = ['genimage_val', 'genimage_sd14']   # diffseg30k and sidset: verified on 12 Nov 2026
CU_PER_HOUR = None      # optional: the CPU runtime's rate from Colab's resource panel
RESERVE_GB = 5.0        # always left free on Drive and on the local disk

T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
paths.datasets.mkdir(parents=True, exist_ok=True)
paths.local_data.mkdir(parents=True, exist_ok=True)
disk_before = {'drive_free_gb': round(free_bytes(paths.datasets) / 1e9, 1),
               'local_free_gb': round(free_bytes(paths.local_data) / 1e9, 1)}
print('free space:', disk_before)
for key in DATASETS:
    print(f'{key:<14} -> {REGISTRY[key].folder:<28} {REGISTRY[key].source}')

INFO:eviground.env:session: python 3.13.18 | torch 2.14.1+cpu | gpu none (None GB) | drive mounted=True free=14.6 GB | commit ad3b07d | CU/h None


free space: {'drive_free_gb': 14.6, 'local_free_gb': 15.4}
genimage_val   -> 01_GenImage/Validation       kaggle:vtphatt2/GenImage-Validation (5 generators already on Drive as loose files)
genimage_sd14  -> 01_GenImage/SD_v1.4_subset   kaggle:vtphatt2/GenImage-stable-diffusion-v1-4


In [8]:
# Cell 3: ensure the datasets (resumable; re-run this cell after a disconnect)
from eviground.data.download import ensure_data

if DRY:   # fake Kaggle / Hugging Face serving tiny synthetic archives
    from eviground.data.fake_sources import (dry_run_overrides, make_fake_world,
                                             make_genimage_validation)
    val_dir = paths.datasets / REGISTRY['genimage_val'].folder
    n_val = make_genimage_validation(val_dir) if not val_dir.exists() else sum(
        1 for p in val_dir.rglob('*') if p.suffix.lower() in ('.png', '.jpeg'))
    sources, overrides = make_fake_world(paths.root.parent / 'fake_remote'), dry_run_overrides(n_val)
    sid_dir = paths.datasets / REGISTRY['sidset'].folder
    if not (sid_dir / 'train_manifest.csv').exists():   # SID-Set as the old downloader left it
        from eviground.data.fake_sources import make_sid_legacy
        make_sid_legacy(sid_dir, overrides['sidset']['caps'])
    DATASETS = ['genimage_val', 'diffseg30k', 'sidset', 'genimage_sd14']
else:
    sources, overrides = None, None

report = ensure_data(datasets=DATASETS, paths=paths, sources=sources, reserve_gb=RESERVE_GB,
                     raise_errors=False, overrides=overrides)
for key, entry in report.items():
    print(f"{key:<14} {entry['status']:<9} {entry.get('error', '')}")

INFO:eviground.data.download:genimage_val: running verify_folder
ERROR:eviground.data.download:genimage_val: InsufficientSpaceError: the GenImage Validation archive needs 25.4 GB on the local disk (/content/data) and a 5 GB reserve must stay free, but only 15.8 GB is free. Nothing was written for this step. Free at least 14.6 GB on the local disk and run it again; finished parts are kept and skipped.
INFO:eviground.data.download:genimage_sd14: running kaggle_subset
INFO:eviground.data.download:SD v1.4 archive: 335997 members
ERROR:eviground.data.download:genimage_sd14: InsufficientSpaceError: the GenImage SD v1.4 subset needs 23.1 GB on Drive (/content/drive/MyDrive/Paper 1/datasets) and a 5 GB reserve must stay free, but only 15.0 GB is free. Nothing was written for this step. Free at least 13.1 GB on Drive and run it again; finished parts are kept and skipped.


genimage_val   error     InsufficientSpaceError: the GenImage Validation archive needs 25.4 GB on the local disk (/content/data) and a 5 GB reserve must stay free, but only 15.8 GB is free. Nothing was written for this step. Free at least 14.6 GB on the local disk and run it again; finished parts are kept and skipped.
genimage_sd14  error     InsufficientSpaceError: the GenImage SD v1.4 subset needs 23.1 GB on Drive (/content/drive/MyDrive/Paper 1/datasets) and a 5 GB reserve must stay free, but only 15.0 GB is free. Nothing was written for this step. Free at least 13.1 GB on Drive and run it again; finished parts are kept and skipped.


In [4]:
# Cell 4: save the record to Drive (runs/checkpoints/cp1_download_<datasets>.json)
import datetime

out = paths.runs / 'checkpoints'
out.mkdir(parents=True, exist_ok=True)
hours = (time.time() - T0) / 3600
record = {
    'checkpoint': 'CP1-download',
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'dry_run': DRY,
    'branch': BRANCH,
    'datasets': DATASETS,
    'session': info,
    'disk_before': disk_before,
    'disk_after': {'drive_free_gb': round(free_bytes(paths.datasets) / 1e9, 1),
                   'local_free_gb': round(free_bytes(paths.local_data) / 1e9, 1)},
    'wall_time_h': round(hours, 3),
    'cu_per_hour': CU_PER_HOUR,
    'cu_estimate': round(hours * CU_PER_HOUR, 3) if CU_PER_HOUR else None,
    'report': report,
}
path = out / f"cp1_download_{'+'.join(DATASETS)}.json"
path.write_text(json.dumps(record, indent=1, default=str))
print('saved', path, '| wall time', round(hours * 60, 1), 'min | disk after', record['disk_after'])
failed = [k for k, v in report.items() if v['status'] == 'error']
assert not failed, f'not finished: {failed} (see the errors above; re-run Cell 3 to resume)'

saved /content/drive/MyDrive/Paper 1/runs/checkpoints/cp1_download_genimage_val+genimage_sd14.json | wall time 32.0 min | disk after {'drive_free_gb': 14.6, 'local_free_gb': 15.4}


AssertionError: not finished: ['genimage_val', 'genimage_sd14'] (see the errors above; re-run Cell 3 to resume)